# Global Pollution Classification (Low, Medium, High)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
df = pd.read_csv("Global_Pollution_Analysis.csv")
df.head()

In [ ]:
# Handle missing values
df.fillna(df.mean(numeric_only=True), inplace=True)

for col in df.select_dtypes(include='object'):
    df[col].fillna(df[col].mode()[0], inplace=True)

In [ ]:
# Encode categorical columns
le = LabelEncoder()
for col in df.select_dtypes(include='object'):
    df[col] = le.fit_transform(df[col])

In [ ]:
# Feature Engineering
if 'Population' in df.columns:
    df['Energy_per_Capita'] = df['Energy_Consumption'] / df['Population']

# Create Pollution Category
def pollution_category(x):
    if x < df['Air_Pollution_Index'].quantile(0.33):
        return "Low"
    elif x < df['Air_Pollution_Index'].quantile(0.66):
        return "Medium"
    else:
        return "High"

df['Pollution_Level'] = df['Air_Pollution_Index'].apply(pollution_category)


In [ ]:
# Encode target
df['Pollution_Level'] = LabelEncoder().fit_transform(df['Pollution_Level'])

In [ ]:
# Feature Scaling
scaler = StandardScaler()
features = df.drop(['Pollution_Level'], axis=1)
features = pd.DataFrame(scaler.fit_transform(features), columns=features.columns)

X = features
y = df['Pollution_Level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [ ]:
# Naive Bayes (Multinomial requires non-negative values)
X_train_nb = X_train - X_train.min()
X_test_nb = X_test - X_test.min()

nb = MultinomialNB()
nb.fit(X_train_nb, y_train)
nb_pred = nb.predict(X_test_nb)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print(classification_report(y_test, nb_pred))

In [ ]:
# KNN with tuning
param_grid = {'n_neighbors': range(3,10)}
knn = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
knn.fit(X_train, y_train)

knn_pred = knn.predict(X_test)

print("Best K:", knn.best_params_)
print("KNN Accuracy:", accuracy_score(y_test, knn_pred))
print(classification_report(y_test, knn_pred))

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=6, min_samples_split=5)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print(classification_report(y_test, dt_pred))

In [ ]:
# Confusion Matrix
sns.heatmap(confusion_matrix(y_test, dt_pred), annot=True, fmt='d')
plt.title("Decision Tree Confusion Matrix")
plt.show()

## Insights

- Multinomial Naive Bayes works after converting features to non-negative values.
- KNN provides flexible decision boundaries but depends on K value.
- Decision Tree gives good interpretability and stable performance.

### Recommendations:
- Use Decision Tree for policy-making (interpretable).
- Use KNN when higher accuracy is required.
- Focus on countries with high pollution for energy recovery programs.
